# Notebook B: Database-Stored Model Embeddings

This notebook demonstrates vector search using embedding models already stored in Oracle AI Database (for example `ALL_MINILM_L12_V2`).

Flow:
- discover available database models
- generate embeddings with `provider=database`
- store vectors and run similarity search


In [ ]:
import os
import json
import re
import oracledb
from dotenv import dotenv_values


## 1) Load DB configuration from `/home/.env`

In [ ]:
ENV_PATH = os.getenv('LAB_ENV_FILE', '/home/.env')
env = dotenv_values(ENV_PATH) if os.path.exists(ENV_PATH) else {}

DB_USER = os.getenv('DB_USER') or env.get('USERNAME') or 'ADMIN'
DB_PASSWORD = os.getenv('DB_PASSWORD') or env.get('DBPASSWORD')
DB_HOST = os.getenv('DB_HOST', 'aidbfree')
DB_PORT = os.getenv('DB_PORT', '1521')
DB_SERVICE = os.getenv('DB_SERVICE', 'FREEPDB1')
DB_DSN = os.getenv('DB_DSN', f'{DB_HOST}:{DB_PORT}/{DB_SERVICE}')
PREFERRED_DB_MODEL = os.getenv('DB_EMBED_MODEL', 'ALL_MINILM_L12_V2').upper()

print('ENV file:', ENV_PATH if os.path.exists(ENV_PATH) else 'not found')
print('DB user:', DB_USER)
print('DB dsn :', DB_DSN)
print('Preferred DB model:', PREFERRED_DB_MODEL)

if not DB_PASSWORD:
    raise ValueError('DB password not found. Set DB_PASSWORD or DBPASSWORD in /home/.env')


## 2) Connect and discover stored models

In [ ]:
conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cur = conn.cursor()

cur.execute('select user from dual')
print('Connected as:', cur.fetchone()[0])

cur.execute('''
    SELECT model_name
    FROM user_mining_models
    ORDER BY model_name
''')
models = [row[0] for row in cur.fetchall()]
print('Models in USER_MINING_MODELS:', models)

if not models:
    raise RuntimeError('No models found in USER_MINING_MODELS. Provision an ONNX model first.')

MODEL_NAME = PREFERRED_DB_MODEL if PREFERRED_DB_MODEL in models else models[0]
print('Selected DB model:', MODEL_NAME)

if not re.match(r'^[A-Z][A-Z0-9_$#]*$', MODEL_NAME):
    raise ValueError(f'Unsafe model identifier: {MODEL_NAME}')


## 3) Determine embedding dimension from the DB model

In [ ]:
db_params = json.dumps({
    'provider': 'database',
    'model': MODEL_NAME,
})

cur.execute('''
    SELECT VECTOR_DIMENSION_COUNT(
        DBMS_VECTOR.UTL_TO_EMBEDDING(:txt, JSON(:params))
    )
    FROM dual
''', {'txt': 'dimension probe', 'params': db_params})

EMBEDDING_DIM = int(cur.fetchone()[0])
print('Embedding dimension:', EMBEDDING_DIM)


## 4) Create table and store in-database embeddings

In [ ]:
TABLE_NAME = 'PRIVATEAI_DOCS_DBMODEL'

try:
    cur.execute(f'DROP TABLE {TABLE_NAME} PURGE')
except oracledb.DatabaseError:
    pass

cur.execute(f'''
    CREATE TABLE {TABLE_NAME} (
        doc_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        title VARCHAR2(200) NOT NULL,
        content CLOB NOT NULL,
        embedding VECTOR({EMBEDDING_DIM}, FLOAT32)
    )
''')

docs = [
    ('Database Model Path', 'Embeddings can be generated directly in Oracle AI Database with a stored ONNX model.'),
    ('Operational Simplicity', 'Keeping model inference in-database can simplify architecture and governance.'),
    ('Vector Search SQL', 'Use VECTOR_DISTANCE with COSINE to rank semantic similarity results.'),
    ('Model Governance', 'Database-stored models can be versioned and controlled with DB privileges.'),
]

inserted = 0
for title, content in docs:
    cur.execute(f'''
        INSERT INTO {TABLE_NAME} (title, content, embedding)
        VALUES (
            :title,
            :content,
            DBMS_VECTOR.UTL_TO_EMBEDDING(:content, JSON(:params))
        )
    ''', {'title': title, 'content': content, 'params': db_params})
    inserted += 1

conn.commit()
print('Inserted rows:', inserted)


## 5) Similarity search with the database model

In [ ]:
query_text = 'Which approach keeps embedding generation inside Oracle Database?'

sql = f'''
SELECT
    title,
    ROUND(1 - VECTOR_DISTANCE(
        embedding,
        DBMS_VECTOR.UTL_TO_EMBEDDING(:query_text, JSON(:params)),
        COSINE
    ), 4) AS similarity,
    SUBSTR(content, 1, 120) AS preview
FROM {TABLE_NAME}
ORDER BY VECTOR_DISTANCE(
    embedding,
    DBMS_VECTOR.UTL_TO_EMBEDDING(:query_text, JSON(:params)),
    COSINE
)
FETCH FIRST 3 ROWS ONLY
'''

cur.execute(sql, {'query_text': query_text, 'params': db_params})
rows = cur.fetchall()

print('Query:', query_text)
for idx, (title, score, preview) in enumerate(rows, 1):
    print(f'{idx}. {title} | score={score}')
    print(f'   {preview}')


In [ ]:
# Optional cleanup
# cur.execute(f'DROP TABLE {TABLE_NAME} PURGE')
# conn.commit()


In [ ]:
cur.close()
conn.close()
print('Connection closed.')
